# EmotionLoRA — Train Empathy Adapter

**Runtime required:** T4 GPU (Runtime → Change runtime type → T4 GPU)

This notebook trains the first LoRA adapter (empathy) on Mistral 7B and pushes it to HuggingFace Hub.

Steps:
1. Install dependencies
2. Clone the repo
3. Set HF token
4. Prepare training data
5. Train the adapter
6. Push to HuggingFace Hub
7. Verify

## Cell 1 — Install dependencies

Unsloth must be installed before any other imports. This takes ~2 minutes on first run.

In [ ]:
# Install Unsloth (handles its own CUDA-compatible torch install)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q

# Install remaining dependencies
!pip install trl transformers peft datasets python-dotenv -q

print('Dependencies installed.')

## Cell 2 — Clone the repo

Pulls the latest code from GitHub into `/content/emotionlora`.

In [ ]:
import os

REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/YOUR_REPO_NAME.git"  # <- update this
REPO_DIR = "/content/emotionlora"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls

## Cell 3 — Set HuggingFace token

Paste your HF write token below. This is used by `push_to_hub.py` to upload the adapter.

In [ ]:
# Write the token to a .env file so push_to_hub.py can read it
HF_TOKEN = "hf_your_token_here"  # <- paste your token here

with open(".env", "w") as f:
    f.write(f"HF_TOKEN={HF_TOKEN}\n")

print(".env written.")

## Cell 4 — Prepare training data

Converts `data/empathy_examples.jsonl` → HuggingFace Dataset with Alpaca format.

In [ ]:
!python data/prepare.py --emotion empathy

## Cell 5 — Train the adapter

Loads Mistral 7B (4-bit), attaches LoRA, trains for 60 steps (~5–10 minutes on T4).

Watch the loss numbers — they should decrease over steps. A final loss around 1.0–1.5 is healthy for this dataset size.

In [ ]:
!python training/train_adapter.py --emotion empathy

## Cell 6 — Push adapter to HuggingFace Hub

Uploads only the LoRA adapter weights (a few MB) — not the full base model.

In [ ]:
!python training/push_to_hub.py --emotion empathy

## Cell 7 — Verify

Confirms the adapter is live on the Hub and lists the files that were uploaded.

In [ ]:
from huggingface_hub import list_repo_files
import json

registry = json.load(open("adapters/registry.json"))
repo_id = registry["empathy"]["repo_id"]

print(f"Checking repo: {repo_id}\n")
files = list(list_repo_files(repo_id))

if files:
    print("Files on Hub:")
    for f in files:
        print(f"  {f}")
    print(f"\nAdapter live at: https://huggingface.co/{repo_id}")
else:
    print("No files found — push may have failed. Check the output of Cell 6.")